In [ ]:
# Cell 1 - Load Data and Find Optimal Threshold for x10
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_curve

print("="*80)
print("UNLOCKING X10 MODEL WITH OPTIMAL THRESHOLD (MEAN AGGREGATION)")
print("="*80)

# 1. Load the saved x10 data
RESULTS_PATH = Path("./5 Class Evaluation Results")
with open(RESULTS_PATH / 'x10_patient_metrics_mean.json', 'r') as f:
    x10_data = json.load(f)

# Extract raw probabilities and true labels
probs = np.array(x10_data['patient_probabilities'])
labels = np.array(x10_data['patient_labels'])
old_accuracy = x10_data['accuracy']

# 2. Calculate the ROC curve to find all possible thresholds
fpr, tpr, thresholds = roc_curve(labels, probs)

# 3. Find the Optimal Threshold using Youden's J statistic
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"Old Threshold: 0.5000 (Accuracy: {old_accuracy:.4f})")
print(f"NEW Optimal Threshold: {optimal_threshold:.4f}")

# 4. Recalculate predictions using the NEW threshold
new_preds = (probs >= optimal_threshold).astype(int)

# 5. Calculate new metrics
new_acc = accuracy_score(labels, new_preds)
new_cm = confusion_matrix(labels, new_preds)
new_prec, new_rec, new_f2, _ = precision_recall_fscore_support(labels, new_preds, average=None, labels=[0, 1], beta=2.0)

print(f"\n┌─ NEW PATIENT-LEVEL METRICS (Optimal Threshold)")
print(f"│  Accuracy: {new_acc:.4f}  <-- Did it beat {old_accuracy:.4f}?")
print(f"│")
print(f"│  MF Class:")
print(f"│    Sensitivity: {new_rec[0]:.4f}")
print(f"│    F2-Score:    {new_f2[0]:.4f}")
print(f"│")
print(f"│  Non-MF Class:")
print(f"│    Sensitivity: {new_rec[1]:.4f}")
print(f"│    F2-Score:    {new_f2[1]:.4f}")
print(f"└─")

print(f"\n  NEW Patient-Level Confusion Matrix:")
print(f"                 Predicted MF  Predicted Non-MF")
print(f"    Actual MF         {new_cm[0,0]:6d}         {new_cm[0,1]:12d}")
print(f"    Actual Non-MF     {new_cm[1,0]:6d}         {new_cm[1,1]:12d}")